# ACCESS-AIS3 - Development workflow

This workbook presents v0.1 of the ACCESS-AIS3 Antarctic model configuration. Development is completed in this workbook and will be converted/exported to an executable script once complete.

In [ ]:
import pyissm
import ccdtools
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import os

# TIPS:
## Compute Instance
Some datasets (e.g. racmo2.4p1_monthly_11km_1979-2023) in `ccdtools` require considerble memory. When loading these datasets is necessary (e.g. during "param") a compute instance with large memory is required. A "custom" compute size of `cpus=8 mem=128G` is recommended.

# TODO:
## Boundary conditions

Here, we specify Neumann BCs at the ice-front. Because there is ocean included in the model, the ice-front extends around the entire continent. We manually specify 0 m/yr velocity Dirichlet BCs on the mesh boundary for completeness. At various stages of the initialisation, it may be necessary to manually add additional Dirichlet BCs internally to help stabilise the model. If this is necessary, the parameterisation step should be updated to specify such Dirichlet BCs upfront.

### Rheology inversion
For the rheology inversion, we extract the floating ice nodes, including the ice-front. This ensures that we maintain the Neumann BCs along the ice-front. By default, `extract()` applies Dirichlet BCs around the boundary, using the observed velocities. In this case, this applies Dirichlet BCs at the outside edge of elements on the ice-front (i.e. enforces a velocity at the ice-front). This behaves okay for the rheology inversion.

### Friction inversion
The friction inversion is not behaving well right now. This _could_ be influenced by the BCs. I have tried a couple of different things:
- `AIS3_ssa_friction_inv_sensit`: Extracting all ice, including the ice-front elements, and setting Neumann ONLY on the ice-front of floating ice, and Neumann AND Dirichlet (on the outside edge of ice-front elements) on grounded ice-front regions. Manually adjusting the BCs in this way ensures that the Neumann ice-front BCs are applied correctly (continguous elements). The model runs successfully using this approach.
- `AIS3_ssa_friction_inv_sensit_2`: Extracting all ice, including the ice-front elements, and using `pyissm.model.bc.marine_ice_sheet_bc()` on the extracted domain. This is equivalent to setting Neumann ONLY on the ice-front of floating ice, and Neumann AND Dirichlet (on the outside edge of ice-front elements) on grounded ice-front regions. However, it doesn't appear to identify a contiguous series of elements along the ice-front. I've confirmed that this behaviour is consistent with MATLAB (and therefore, expected...?). The model fails, with `solver residue too high` errors.
- `AIS3_ssa_friction_inv_sensit_3`: Extracting all ice (excluding the ice-front), using the ice-mask only (i.e., not buffering to include the ice-front elements) and setting Dirichlet EVERYWHERE (i.e., no Neumann BCs anywhere). This model runs successfully using this approach.

**NOTE: These repeat steps can be removed once the BCs are confirmed. They're just included here for testing**

## Relative paths
Update paths for all input data/files (e.g. `domain_file`, `param_file` etc.) to be relative paths so that the model config becomes more portable for other users/locations.

## MIPKIT
To support ISMIP7 efforts, the ISMIP7 working group have released "Antarctica MIPKIT", a datset/file with common datsaets/fields used for model initialisation of ISMIP7 models. While we use the "same" original datasets here, using MIPKIT would ensure consistency with other ISMIP7 efforts. MIPTKIP can be added to the `ccd`, as per this issue: https://github.com/ACCESS-NRI/ccdtools/issues/61. Where possible, individual datasets used here and contained within MIPKIT should be updated to use MIPKIT.

## Inversion sensitivity
### Rheology inversion
The initial rheology inversion sensitivity provides coefficient values that yield reasonable results. However, the coefficient values may not be fully optimised.

### Friction inversion
Most friction inversion tests have focused on changes in BCs and extraction domains (i.e. including the ice-front or not). It might be worth extending the `mask` parsed to `parameter_sensitivity()` to exclude regions of grounded ice where the effective pressure is 0. This would exclude regions such as the Trans Antarctic Mountains.

## Regional Output
Currently, `pyissm.model.io.save_model()` does not support lists of ISSM classes. Since `md.outputdefinition.definitions` is a list of `pyissm.model.classes.regionaloutput()` objects, saving the model to NetCDF currently fails. The model can be marshalled (writted to *.bin), but not to NetCDF. The list handling in `_serialize_arrays_lists` within `pyissm.model.io.save_model()` needs to be updated to handle lists of objects.

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/lb9857/access-ais3/')

# Should plots be generated?
plot = False

# Shuld diagnostics be printed?
diagnostics = True

# Using load_only?
load_only = True

# Define execution directory
execution_dir = '/g/data/au88/lb9857/access-ais3/execution'

# Define location to save final models
model_dir = '/g/data/au88/lb9857/access-ais3/models'

# Define domain_file
domain_file = ('/g/data/au88/lb9857/gitRepos/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/lb9857/gitRepos/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm/2025.11.0']
cluster.np = 32
cluster.memory = 100
cluster.time = 60*48
cluster.login = 'lb9857'
cluster.project = 'au88'

# List all steps for clarity
all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    'ssa_friction_inv_sensit',
    'extrude',
    'ho_rheology_floating_inv_sensit',
    'ho_rheology_floating_inv_lcurve',
    'h0_friction_inv_sensit'
]

# Define steps to run
# steps = ['process_domain']
# steps = ['mesh']
# steps = ['param']
# steps = ['ssa_rheology_floating_inv_sensit']
# steps = ['ssa_rheology_floating_inv_lcurve']
# steps = ['ssa_friction_inv_sensit'] # SCHOOF; Coupling = 3; Extract ice (and ice-front)
# steps = ['ssa_friction_inv_sensit_2'] # SCHOOF; Coupling = 3; Extract ice only; Marince Ice Sheet BCs
# steps = ['ssa_friction_inv_sensit_3'] # SCHOOF; Coupling = 3; Extract ice only; Dirichlet everywhere (extract default)


In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

In [ ]:
## ------------------------------------
## Process domain file
## ------------------------------------

if 'process_domain' in steps:

    print("-------------------------------------------------------------")
    print(f" PROCESSING DOMAIN FILE"                                     )
    print("-------------------------------------------------------------")

    # Buffer coastline polygon by 100 km
    print(f" - Buffering coastline...")
    coastline_100km_buffer = measures_coastline.buffer(100000)

    # Write buffered extent to file for use as model domain
    print(f" - Saving to file...")
    pyissm.tools.exp.gdf_to_exp(coastline_100km_buffer, '/g/data/au88/gitRepos/ACCESS-AIS3/assets/ais_domain.exp')


In [ ]:
## ------------------------------------
## Create mesh
## ------------------------------------
if 'mesh' in steps:

    print("-------------------------------------------------------------")
    print(f" GENERATING MESH"                                            )
    print("-------------------------------------------------------------")

    # Create empty model with initial 10e3 resolution mesh
    md = pyissm.model.mesh.triangle(pyissm.model.Model(), domain_file, 10e3)
    
    # Remesh the model twice to refine based on velocity and bedmachine mask
    for i in range(2):

        print(f"REFINEMENT ITERATION: {i+1}")

        # Interpolate velocities onto mesh
        print(f"\n-- Interpolating MEaSURES v2 Velocities...")
        vx = pyissm.data.interp.xr_to_mesh(velocity_data, 'VX', md.mesh.x, md.mesh.y)
        vy = pyissm.data.interp.xr_to_mesh(velocity_data, 'VY', md.mesh.x, md.mesh.y)
        vel = np.sqrt(vx**2 + vy**2)

        # Interpolate ice mask onto mesh
        print(f"\n-- Interpolating Bedmachine v3 Ice Mask...")
        mask = pyissm.data.interp.xr_to_mesh(bedmachine_data, 'mask', md.mesh.x, md.mesh.y, interpolation_type = 'nearest')

        # Fill NaN values and set to 0 ice-free areas (and ocean, but over-ridden below)
        print(f"\n-- Set Velocity to 0 where NaNs exist or mask < 2...")
        vel[np.isnan(vel) | (mask < 2)] = 0.0

        # Set velocity to 0 in ocean areas to prevent sharp velocity gradient at ice-front impacting meshing
        print(f"\n-- Set Velocity to NaN in ocean areas...")
        vel[(mask == 0)] = np.nan

        if diagnostics:
            print(f"\nVELOCITY DIAGNOSTICS:")
            print(f"    Max velocity: {np.nanmax(vel):.2f} m/yr")
            print(f"    Min velocity: {np.nanmin(vel):.2f} m/yr")

            print(f"\nMASK DIAGNOSTICS:")
            unique_vals, counts = np.unique(mask, return_counts=True)
            for val, count in zip(unique_vals, counts):
                print(f"    Value {val}: {count} occurrences")

        if plot:
            pyissm.plot.plot_model_field(md, vel, cmap = 'PuOr',show_cbar = True, cbar_kwargs = {'label': 'Velocity (m/a)'}); plt.show(block = False)
            pyissm.plot.plot_model_field(md, mask, show_cbar = True, cbar_kwargs = {'label': 'Ice Mask'}); plt.show(block = False)        

        # Define min/max vertex lengths in specific regions
        print(f"\n-- Setting min/max vertex dimensions...")
        hmax_v = np.full(md.mesh.numberofvertices, np.nan)
        hmin_v = np.full(md.mesh.numberofvertices, np.nan)
    
        hmax_v[(vel > 50) & (mask == 2)] = 1500 # Max length on fast-flowing grounded ice
        hmin_v[(mask == 3)] = 500 # Min length on ice shelves
        hmax_v[(mask == 3)] = 5000 # Max length on ice shelves
        hmax_v[(mask == 0)] = 5000 # Max length in ocean

        # Adjust mesh with specified metrics
        print(f"\n-- Remeshing with specified metrics...")
        md = pyissm.model.mesh.bamg(md,
                                    hmin = 50,
                                    hmax = 50e3,
                                    hmaxVertices = hmax_v,
                                    hminVertices = hmin_v,
                                    maxnbv = 2e6,
                                    field = vel,
                                    err = 1,
                                    gradation = 1.2)
        
        # Remove bamg private data to allow additional remeshes
        md.private.bamg = {}
    
        if diagnostics:
            print(f"\nMESH DIAGNOSTICS:")
            print(f"   Number of elements: {md.mesh.numberofelements}")
            print(f"   Number of vertices: {md.mesh.numberofvertices}")
    
    # Set georefernce information
    [md.mesh.lat, md.mesh.long] = pyissm.tools.general.xy_to_ll(md.mesh.x, md.mesh.y, -1)
    md.mesh.epsg = 3031
    
    print(f"\nFinal mesh: {md.mesh.numberofvertices} nodes; {md.mesh.numberofelements} elements")
    
    if plot:
        areas = pyissm.model.mesh.get_element_areas_volumes(md.mesh.elements, md.mesh.x, md.mesh.y)
        # A = l^2 * sqrt(3) / 4 is area for equilateral triangle
        # np.sqrt(A*2) / 1e3 is the rough approximation of element edge length in km
        fig, ax = pyissm.plot.plot_model_field(md,
                                               np.sqrt(areas*2)/1e3,
                                               show_cbar = True,
                                               vmin = 0.25,
                                               vmax = 10,
                                               plot_data_on='elements',
                                               cmap = 'plasma_r',
                                               cbar_kwargs = {'label': 'Approx. element edge length (km)'})
        ax.set_title('Final Mesh: Velocity-adapted w/ 2 refinement passes')
        plt.show(block = False)

        print(f"\nSaving model to {model_dir}/AIS3_mesh.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_mesh.nc')

In [ ]:
## ------------------------------------
## Parameterise model
## ------------------------------------
if 'param' in steps:

    print("-------------------------------------------------------------")
    print(f" PARAMETERIZING MODEL"                                       )
    print("-------------------------------------------------------------")

    print(f"-- Loading model mesh...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_mesh.nc')

    print(f"-- Parameterising model using {param_file}...")
    md = pyissm.model.param.parameterize(md, param_file)

    print(f"-- Set flow equation to SSA...")
    md = pyissm.model.param.set_flow_equation(md, SSA = 'all')

    print(f"-- Setting Boundary Conditions...")
    # -------- Set Stress Balance BCs --------
    ## Initialize empty fields
    md.stressbalance.spcvx = np.full(md.mesh.numberofvertices, np.nan)
    md.stressbalance.spcvy = np.full(md.mesh.numberofvertices, np.nan)
    md.stressbalance.spcvz = np.full(md.mesh.numberofvertices, np.nan)

    ## Find nodes on the boundary
    boundary_nodes = md.mesh.vertexonboundary.astype(bool)

    ## Set 0 m/yr velocity dirichlet BCs on the boundary
    md.stressbalance.spcvx[boundary_nodes] = 0
    md.stressbalance.spcvy[boundary_nodes] = 0
    md.stressbalance.spcvz[boundary_nodes] = 0
    
    md.stressbalance.referential = np.nan * np.ones((md.mesh.numberofvertices, 6))
    md.stressbalance.loadingforce = np.zeros((md.mesh.numberofvertices, 3))

    # -------- Set thermal Balance BCs --------
    md.thermal.spctemperature = md.initialization.temperature.copy()
  

    if diagnostics:
        print(f"\nMASK DIAGNOSTICS:")
        print(f" - Ice Levelset:")        
        unique_vals, counts = np.unique(md.mask.ice_levelset, return_counts=True)
        for val, count in zip(unique_vals, counts):
            print(f"    Value {val}: {count} occurrences")

        print(f" Ocean Levelset (Binary):")
        ocean_binary = md.mask.ocean_levelset <= 0
        unique_vals, counts = np.unique(ocean_binary, return_counts=True)
        for val, count in zip(unique_vals, counts):
            print(f"    Value {val}: {count} occurrences")

        print(f"\nGEOMETRY DIAGNOSTICS:")
        print(f" - Surface elevation:")
        print(f"   min = {np.min(md.geometry.surface):.2f} m")
        print(f"   max = {np.max(md.geometry.surface):.2f} m")
        print(f" - Bed elevation:")
        print(f"   min = {np.min(md.geometry.bed):.2f} m")
        print(f"   max = {np.max(md.geometry.bed):.2f} m")
        print(f" - Thickness:")
        print(f"   min = {np.min(md.geometry.thickness):.2f} m")
        print(f"   max = {np.max(md.geometry.thickness):.2f} m")

        print(f"VELOCITY DIAGNOSTICS:")
        print(f"   Min observed vx: {np.min(md.inversion.vx_obs):.2f} m/yr")
        print(f"   Max observed vx: {np.max(md.inversion.vx_obs):.2f} m/yr")
        print(f"   Min observed vy: {np.min(md.inversion.vy_obs):.2f} m/yr")
        print(f"   Max observed vy: {np.max(md.inversion.vy_obs):.2f} m/yr")
        print(f"   Min observed vel: {np.min(md.inversion.vel_obs):.2f} m/yr")
        print(f"   Max observed vel: {np.max(md.inversion.vel_obs):.2f} m/yr")

        print(f"INITIAL PRESSURE DIAGNOSTICS:")
        print(f"   Min initial pressure: {np.min(md.initialization.pressure):.2f} Pa")
        print(f"   Max initial pressure: {np.max(md.initialization.pressure):.2f} Pa")

        print(f"GEOTHERMAL HEAT FLOW DIAGNOSTICS:")
        print(f"   Min geothermal heat flux: {np.min(md.basalforcings.geothermalflux):.7f} mW/m2")
        print(f"   Max geothermal heat flux: {np.max(md.basalforcings.geothermalflux):.7f} mW/m2")

        print(f"INITIAL TEMPERATURE DIAGNOSTICS:")
        print(f"   Min initial temp: {np.min(md.initialization.temperature):.2f} K")
        print(f"   Max initial temp: {np.max(md.initialization.temperature):.2f} K")

        print(f"\nSaving model to {model_dir}/AIS3_param.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_param.nc')

In [ ]:
## ------------------------------------
## SSA Rheology Inversion Sensitivity - Floating Ice
## ------------------------------------
if 'ssa_rheology_floating_inv_sensit' in steps:
    
    print("-------------------------------------------------------------")
    print(f" SSA RHEOLOGY INVERSION SENSITIVITY - FLOATING ICE"          )
    print("-------------------------------------------------------------")

    # Define directory for all inversion sensitivity outputs
    sensit_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_sensit'
    
    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    
    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Defining inversion parameters...")
    md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
    md.inversion.iscontrol = 1
    md.inversion.control_parameters = ['MaterialsRheologyBbar']
    md.inversion.min_parameters = pyissm.tools.materials.cuffey(273.15 - 0) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.max_parameters = pyissm.tools.materials.cuffey(273.15 - 70) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.maxsteps = 500
    md.inversion.maxiter = 200

    # Remove icebergs
    print(f"-- Removing icebergs from ice levelset...")
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)
    
    print(f"-- Extracting floating ice only (including ice-front)...")
    ocean_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ocean_levelset)
    ice_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract((ocean_elements < 1) & (ice_elements < 1))

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    print(f"-- Setting-up coefficient grid...")
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [1, 10, 100, 1000],
         103: [1, 10, 100, 1000]})
    
    print(f"-- Defining mask to exclude 0 velocity & non-ice nodes from inversion...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ice_levelset < 0)

    if load_only:
        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = False,
            load_only = True,
            global_mask = mask)

        # Process diagnostics
        print(f"-- Processing inversion parameter sensitivity...")
        diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir = sensit_dir)        

        # Normalize diagnostics (0-1)
        diagnostics_norm = pyissm.inversion.sensitivity.normalize_diagnostics(diagnostics,
                                                                             columns = ['vel_rmse', 'mean_gradient_magnitude'])

        # Calculate "overall" diagnostic
        diagnostics_norm['overall'] = diagnostics_norm['vel_rmse_norm'] + diagnostics_norm['mean_gradient_magnitude_norm']
        
        # Plot diagnostic heatmaps
        fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize = (15, 8), constrained_layout = True)
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax1, value = 'vel_rmse')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax2, value = 'ratio_101_103')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax3, value = 'mean_gradient_magnitude')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax4, value = 'cost_total')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax5, value = 'positive_residual_fraction')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics_norm, x = 'cf101', y = 'cf103', ax = ax6, value = 'overall')
        plt.savefig(f'{sensit_dir}/diagnostic_heatmaps.png')

        # Summarise the best run
        best_row = diagnostics_norm.loc[diagnostics_norm['overall'].idxmax()]
        print(f"The best run_id is: {best_row['run_id']}. This uses the following coefficient values:")
        print(best_row.filter(regex=r'^cf'))        
        
    else:
        print(f"-- Running inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = True,
            load_only = False,
            global_mask = mask)

In [ ]:
# ------------------------------------
## SSA Rheology Inversion L-Curve - Floating Ice
## ------------------------------------
if 'ssa_rheology_floating_inv_lcurve' in steps:
    
    print("-------------------------------------------------------------")
    print(f" SSA RHEOLOGY INVERSION L-CURVE - FLOATING ICE"              )
    print("-------------------------------------------------------------")

    # Define directory for all inversion sensitivity outputs
    sensit_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve'
    
    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    
    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Defining inversion parameters...")
    md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
    md.inversion.iscontrol = 1
    md.inversion.control_parameters = ['MaterialsRheologyBbar']
    md.inversion.min_parameters = pyissm.tools.materials.cuffey(273.15 - 0) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.max_parameters = pyissm.tools.materials.cuffey(273.15 - 70) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.maxsteps = 500
    md.inversion.maxiter = 200

    # Remove icebergs
    print(f"-- Removing icebergs from ice levelset...")
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print(f"-- Extracting floating ice only (including ice-front)...")
    ocean_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ocean_levelset)
    ice_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract((ocean_elements < 1) & (ice_elements < 1))

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # Update execution directory to be step-specific
    os.makedirs(f'{execution_dir}/AIS3_ssa_rheology_floating_inv_lcurve', exist_ok = True)
    mds.cluster.executionpath = f'{execution_dir}/AIS3_ssa_rheology_floating_inv_lcurve'
    
    print(f"-- Setting-up coefficient grid...")
    # Use preferred 101/103 coefficients from sensitivity step
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [100],
         103: [1],
         502: [1e-20, 1e-19, 1e-18, 1e-17, 1e-16, 1e-15, 1e-14, 1e-13, 1e-12]})
    
    print(f"-- Defining mask to exclude 0 velocity & non-ice nodes from inversion...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ice_levelset < 0)
    
    if load_only:

        print(f"-- Loading inversion parameter sensitivity...")
        # Only mask 101 and 103 -- no mask on regularisation.
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = False,
            load_only = True,
            coeff_masks = {101: mask,
                           103: mask})

        print(f"-- Processing inversion parameter sensitivity...")
        diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir = sensit_dir)        

        # Plot L-curve
        fig, ax = pyissm.inversion.plot.plot_lcurve(diagnostics)
        ax.set_title('Floating ice rheology inversion - L-curve analysis')
        plt.savefig(f'{sensit_dir}/lcurve.png')

    else:
        
        print(f"-- Running inversion parameter sensitivity...")
        # Only mask 101 and 103 -- no mask on regularisation.
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve',
            run = True,
            load_only = False,
            coeff_masks = {101: mask,
                           103: mask})


In [ ]:
## ------------------------------------
## SSA Friction Inversion
## ------------------------------------
if 'ssa_friction_inv_sensit' in steps:
    
    print("-------------------------------------------------------------")
    print(f" SSA FRICTION INVERSION SENSITIVITY (SCHOOF)- GROUNDED ICE"  )
    print("-------------------------------------------------------------")

    sensit_dir = f'{model_dir}/AIS3_ssa_friction_inv_sensit'
    
    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results...")
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/run_004_100_1_1e-17/run_004_100_1_1e-17.nc')

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    # # Remove icebergs
    print(f"-- Removing icebergs from ice levelset...")
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)
    
    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Extracting ice only (including ice-front)...")
    # Move ice mask to elements to capture ice-front
    ## NOTE: This sets Dirichlet BCs around the entire boundary (including the inflow and ice-front).
    ## By including all ice-front elements, we also maintain Neumann BCs on the ice-front.
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    # Set only Neuman BCs on the floating ice-front (Neuman with DS Dirichlet constraint elsewhere)
    iceFront = (mds.mask.ice_levelset >= 0) & (mds.mask.ocean_levelset < 0)
    mds.stressbalance.spcvx[iceFront] = np.nan
    mds.stressbalance.spcvy[iceFront] = np.nan
    mds.stressbalance.spcvz[iceFront] = np.nan

    print(f"-- Defining inversion parameters...")
    mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
    mds.inversion.control_parameters = ['FrictionC']
    mds.inversion.min_parameters = np.full(mds.mesh.numberofvertices, 0.05)
    mds.inversion.max_parameters = np.full(mds.mesh.numberofvertices, 10000) ## TODO: Set upper bound once better constrained
    mds.inversion.maxsteps = 500
    mds.inversion.maxiter = 200
    
    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # Update execution directory to be step-specific
    os.makedirs(f'{execution_dir}/AIS3_ssa_friction_inv_sensit', exist_ok = True)
    mds.cluster.executionpath = f'{execution_dir}/AIS3_ssa_friction_inv_sensit'

    # No friction on PURELY floating ice elements
    ocean_elements = mds.mask.ocean_levelset[mds.mesh.elements - 1] # -1 for zero-based indexing
    pos_e = np.where(np.min(ocean_elements, axis=1) < 0)[0]
    flags = np.zeros(mds.mesh.numberofvertices, dtype=bool)
    flags[mds.mesh.elements[pos_e, :] - 1] = True # -1 for zero-based indexing
    mds.friction.C[flags] = 0.05
    mds.inversion.min_parameters[flags] = 0.0
    mds.inversion.max_parameters[flags] = 0.0
    
    mds.transient  = pyissm.model.classes.transient.deactivate_all(mds.transient)

    mds.stressbalance.restol  = 0.01
    mds.stressbalance.reltol  = 0.1
    mds.stressbalance.abstol  = np.nan
    mds.settings.solver_residue_threshold  = 1e-3

    print(f"-- Setting-up coefficient grid...")
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [1, 10, 100, 1000],
         103: [1, 10, 100, 1000]})

    print(f"-- Defining mask to exclude 0 velocity & non-ice nodes from inversion...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ice_levelset < 0)

    if load_only:

        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = False,
            load_only = True,
            global_mask = mask)

        # Process diagnostics
        # print(f"-- Processing inversion parameter sensitivity...")
        # diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir = sensit_dir)        

        # # Normalize diagnostics (0-1)
        # diagnostics_norm = pyissm.inversion.sensitivity.normalize_diagnostics(diagnostics,
        #                                                                      columns = ['vel_rmse', 'mean_gradient_magnitude'])

        # # Calculate "overall" diagnostic
        # diagnostics_norm['overall'] = diagnostics_norm['vel_rmse_norm'] + diagnostics_norm['mean_gradient_magnitude_norm']
        
        # # Plot diagnostic heatmaps
        # fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize = (15, 8), constrained_layout = True)
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax1, value = 'vel_rmse')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax2, value = 'ratio_101_103')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax3, value = 'mean_gradient_magnitude')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax4, value = 'cost_total')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax5, value = 'positive_residual_fraction')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics_norm, x = 'cf101', y = 'cf103', ax = ax6, value = 'overall')
        # plt.savefig(f'{sensit_dir}/diagnostic_heatmaps.png')

        # # Summarise the best run
        # best_row = diagnostics_norm.loc[diagnostics_norm['overall'].idxmax()]
        # print(f"The best run_id is: {best_row['run_id']}. This uses the following coefficient values:")
        # print(best_row.filter(regex=r'^cf'))        


    else:
        print(f"-- Running inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = True,
            load_only = False,
            global_mask = mask)

In [ ]:
## ------------------------------------
## SSA Friction Inversion
## ------------------------------------
if 'ssa_friction_inv_sensit_2' in steps:
    
    print("-------------------------------------------------------------")
    print(f" SSA FRICTION INVERSION SENSITIVITY (SCHOOF)- GROUNDED ICE"  )
    print("-------------------------------------------------------------")

    sensit_dir = f'{model_dir}/AIS3_ssa_friction_inv_sensit_2'
    
    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results...")
    # mds = pyissm.model.io.load_model(f'{execution_dir}/AIS3_ssa_rheology_floating_inv.nc')
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/run_004_100_1_1e-17/run_004_100_1_1e-17.nc')

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    # # Remove icebergs
    print(f"-- Removing icebergs from ice levelset...")
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)
    
    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Extracting ice only (excluding ice-front elements)...")
    mds = md.extract(md.mask.ice_levelset < 0)

    # Set only Neuman BCs on the floating ice-front (Neuman with DS Dirichlet constraint elsewhere)
    mds = pyissm.model.bc.set_marine_ice_sheet_bc(mds)

    print(f"-- Defining inversion parameters...")
    mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
    mds.inversion.control_parameters = ['FrictionC']
    mds.inversion.min_parameters = np.full(mds.mesh.numberofvertices, 0.05)
    mds.inversion.max_parameters = np.full(mds.mesh.numberofvertices, 10000) ## TODO: Set upper bound once better constrained
    mds.inversion.maxsteps = 500
    mds.inversion.maxiter = 200
    
    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # Update execution directory to be step-specific
    os.makedirs(f'{execution_dir}/AIS3_ssa_friction_inv_sensit_2', exist_ok = True)
    mds.cluster.executionpath = f'{execution_dir}/AIS3_ssa_friction_inv_sensit_2'

    # No friction on PURELY floating ice elements
    ocean_elements = mds.mask.ocean_levelset[mds.mesh.elements - 1] # -1 for zero-based indexing
    pos_e = np.where(np.min(ocean_elements, axis=1) < 0)[0]
    flags = np.zeros(mds.mesh.numberofvertices, dtype=bool)
    flags[mds.mesh.elements[pos_e, :] - 1] = True # -1 for zero-based indexing
    mds.friction.C[flags] = 0.05
    # mds.inversion.min_parameters[flags] = 0.0
    # mds.inversion.max_parameters[flags] = 0.0
    
    mds.transient  = pyissm.model.classes.transient.deactivate_all(mds.transient)

    mds.stressbalance.restol  = 0.01
    mds.stressbalance.reltol  = 0.1
    mds.stressbalance.abstol  = np.nan
    mds.settings.solver_residue_threshold  = 1e-3

    print(f"-- Setting-up coefficient grid...")
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [1, 10, 100, 1000],
         103: [1, 10, 100, 1000]})

    print(f"-- Defining mask to exclude 0 velocity & non-ice nodes from inversion...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ice_levelset < 0)

    if load_only:

        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = False,
            load_only = True,
            global_mask = mask)

        # Process diagnostics
        # print(f"-- Processing inversion parameter sensitivity...")
        # diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir = sensit_dir)        

        # # Normalize diagnostics (0-1)
        # diagnostics_norm = pyissm.inversion.sensitivity.normalize_diagnostics(diagnostics,
        #                                                                      columns = ['vel_rmse', 'mean_gradient_magnitude'])

        # # Calculate "overall" diagnostic
        # diagnostics_norm['overall'] = diagnostics_norm['vel_rmse_norm'] + diagnostics_norm['mean_gradient_magnitude_norm']
        
        # # Plot diagnostic heatmaps
        # fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize = (15, 8), constrained_layout = True)
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax1, value = 'vel_rmse')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax2, value = 'ratio_101_103')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax3, value = 'mean_gradient_magnitude')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax4, value = 'cost_total')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax5, value = 'positive_residual_fraction')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics_norm, x = 'cf101', y = 'cf103', ax = ax6, value = 'overall')
        # plt.savefig(f'{sensit_dir}/diagnostic_heatmaps.png')

        # # Summarise the best run
        # best_row = diagnostics_norm.loc[diagnostics_norm['overall'].idxmax()]
        # print(f"The best run_id is: {best_row['run_id']}. This uses the following coefficient values:")
        # print(best_row.filter(regex=r'^cf'))        


    else:
        print(f"-- Running inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = True,
            load_only = False,
            global_mask = mask)

In [ ]:
## ------------------------------------
## SSA Friction Inversion
## ------------------------------------
if 'ssa_friction_inv_sensit_3' in steps:
    
    print("-------------------------------------------------------------")
    print(f" SSA FRICTION INVERSION SENSITIVITY (SCHOOF)- GROUNDED ICE"  )
    print("-------------------------------------------------------------")

    sensit_dir = f'{model_dir}/AIS3_ssa_friction_inv_sensit_3'
    
    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Loading SSA floating rheology inversion results...")
    # mds = pyissm.model.io.load_model(f'{execution_dir}/AIS3_ssa_rheology_floating_inv.nc')
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/run_004_100_1_1e-17/run_004_100_1_1e-17.nc')

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    # # Remove icebergs
    print(f"-- Removing icebergs from ice levelset...")
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)
    
    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Extracting ice only (excluding ice-front)...")
    ## NOTE: This sets Dirichlet BCs around the entire boundary
    mds = md.extract(md.mask.ice_levelset < 0)

    print(f"-- Defining inversion parameters...")
    mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
    mds.inversion.control_parameters = ['FrictionC']
    mds.inversion.min_parameters = np.full(mds.mesh.numberofvertices, 0.05)
    mds.inversion.max_parameters = np.full(mds.mesh.numberofvertices, 10000) ## TODO: Set upper bound once better constrained
    mds.inversion.maxsteps = 500
    mds.inversion.maxiter = 200
    
    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    # Update execution directory to be step-specific
    os.makedirs(f'{execution_dir}/AIS3_ssa_friction_inv_sensit_3', exist_ok = True)
    mds.cluster.executionpath = f'{execution_dir}/AIS3_ssa_friction_inv_sensit_3'

    # No friction on PURELY floating ice elements
    ocean_elements = mds.mask.ocean_levelset[mds.mesh.elements - 1] # -1 for zero-based indexing
    pos_e = np.where(np.min(ocean_elements, axis=1) < 0)[0]
    flags = np.zeros(mds.mesh.numberofvertices, dtype=bool)
    flags[mds.mesh.elements[pos_e, :] - 1] = True # -1 for zero-based indexing
    mds.friction.C[flags] = 0.05
    # mds.inversion.min_parameters[flags] = 0.0
    # mds.inversion.max_parameters[flags] = 0.0
    
    mds.transient  = pyissm.model.classes.transient.deactivate_all(mds.transient)

    mds.stressbalance.restol  = 0.01
    mds.stressbalance.reltol  = 0.1
    mds.stressbalance.abstol  = np.nan
    mds.settings.solver_residue_threshold  = 1e-3

    print(f"-- Setting-up coefficient grid...")
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [1, 10, 100, 1000],
         103: [1, 10, 100, 1000]})

    print(f"-- Defining mask to exclude 0 velocity & non-ice nodes from inversion...")
    mask = (mds.inversion.vel_obs > 0) & (mds.mask.ice_levelset < 0)

    if load_only:

        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = False,
            load_only = True,
            global_mask = mask)

        # Process diagnostics
        # print(f"-- Processing inversion parameter sensitivity...")
        # diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir = sensit_dir)        

        # # Normalize diagnostics (0-1)
        # diagnostics_norm = pyissm.inversion.sensitivity.normalize_diagnostics(diagnostics,
        #                                                                      columns = ['vel_rmse', 'mean_gradient_magnitude'])

        # # Calculate "overall" diagnostic
        # diagnostics_norm['overall'] = diagnostics_norm['vel_rmse_norm'] + diagnostics_norm['mean_gradient_magnitude_norm']
        
        # # Plot diagnostic heatmaps
        # fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize = (15, 8), constrained_layout = True)
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax1, value = 'vel_rmse')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax2, value = 'ratio_101_103')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax3, value = 'mean_gradient_magnitude')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax4, value = 'cost_total')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax5, value = 'positive_residual_fraction')
        # pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics_norm, x = 'cf101', y = 'cf103', ax = ax6, value = 'overall')
        # plt.savefig(f'{sensit_dir}/diagnostic_heatmaps.png')

        # # Summarise the best run
        # best_row = diagnostics_norm.loc[diagnostics_norm['overall'].idxmax()]
        # print(f"The best run_id is: {best_row['run_id']}. This uses the following coefficient values:")
        # print(best_row.filter(regex=r'^cf'))        


    else:
        print(f"-- Running inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = sensit_dir,
            run = True,
            load_only = False,
            global_mask = mask)

# PSEUDO-CODE BELOW

Code blocks below are untested, but might be useful in the remaining development steps. The general structure of these steps matches the proposed config here: https://github.com/ACCESS-NRI/ACCESS-AIS3/issues/1.

**NOTE: Once the model parameterization (including BCs) is finalised, it may be possible to go straight to HO inversions, and not include the above initial SSA inversions.**

TODO:
- Update paths to preferred models from inversion sensitivity steps
- Update `pyissm.model.io.save_model()` to support lists of model classes. This is necessary for `md.outputdefinition.definitions` which contains a list of `pyissm.model.classes.regionaloutput()` classes. The model can be marshalled successfully, but it is not saved successfully.

# 1. Extrdue the model to 3D.

In [ ]:
## ------------------------------------
## Extrude model
## ------------------------------------
if 'extrude' in steps:

    print("-------------------------------------------------------------")
    print(f" EXTRUDING MODEL"                                            )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    ## LOAD & REPLACE INVERTED FIELDS
    
    print(f"-- Loading SSA floating rheology inversion results...")
    # TODO: Update path to preferred model
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/run_004_100_1_1e-17/run_004_100_1_1e-17.nc')

    print(f"-- Updating rheology field from inversion results...")
    md.materials.rheology_B[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    print(f"-- Loading SSA friction inversion results...")
    # TODO: Update path to preferred model
    mds = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_friction_inv_sensit/run_001_1_1/run_001_1_1.nc')
    
    print(f"-- Updating friction field from inversion results...")
    md.friction.C[mds.mesh.extractedvertices - 1] = mds.results.StressbalanceSolution.FrictionC # Note: -1 for zero-based indexing
    
    
    ## EXTRUDE MODEL & UPDATE FLOW EQUATION
    
    print(f"-- Extruding model to 3D...")
    md = md.extrude(10, 1.1)
    
    print(f"-- Set flow equation to HO...")
    md = pyissm.model.param.set_flow_equation(md, HO = 'all')

    if not load_only:
        print(f"\nSaving model to {model_dir}/AIS3_extrude.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_extrude.nc')

# 2. HO Floating ice rheology inversion

Perform HO rheology for floating ice. 

It's unlikely that the `cost_function_coefficients` selected through the SSA sensitivity and L-curve steps hold true for HO inversion, so the sensitivity and L-curve steps are likely required again here. Reuse code from above steps.

# 3. HO Thermal simulation

Perform HO thermal simulation.

# 4. HO Friction Inversion

Perform HO friction inversion.

It's unlikely that the `cost_function_coefficients` selected through the SSA sensitivity and L-curve steps hold true for HO inversion, so the sensitivity and L-curve steps are likely required again here. Reuse code from above steps.

# 5. Relaxation (100-years)

After model initialization, perform a 100-year transient simulation for relaxation. Model output should be as follows:
- Output every 5-years
- Output for regional drainage basins (based on MEaSUREs v2 boundaries)
- Monthly timestep

In order to generate regional output, we must use `regionaloutput` to define various `outputdefinitions`. This takes a long time for multiple (and large) regions, so this is separated to a stand-alone step so the masks can be saved in the model NetCDF file, and not only writted when the model is marshalled. **NOTE: `pyissm.model.io.save_model()` currently doesn't support this (saving lists of objects). See TODO item above.**



In [ ]:
if 'generate_regional_masks' in steps:

    # ---------------------------------
    # LOAD SPATIAL DATA
    # ---------------------------------
    print(f"-- Loading IMBIE Basin spatial data...")
    # Load the IMBIE basins
    imbie = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'basins_imbie')

    # Remove islands polygons
    imbie = imbie[imbie["Regions"] != "Islands"].reset_index(drop=True)

    # ---------------------------------
    # LOAD MODEL
    # ---------------------------------

    # Load the model
    print(f"-- Loading initialized model...")
    md = pyissm.model.io.load_model('OUTPUT FROM HO FRICTION INVERSION')
    
    # ---------------------------------
    # SETUP REGIONAL OUTPUTS
    # ---------------------------------

    # Define desired output metrics for all regions
    metrics = [
        "IceVolume",
        "IceVolumeAboveFloatation",
        "GroundedArea",
        "FloatingArea",
        "IceMass",
        "GroundinglineMassFlux"
    ]

    # Initialise definitions and request outputs
    definitions = []
    requested_outputs = ["default"]

    # Append metrics to requested_outputs so these metrics are returned for the whole domain (as well as regionally)
    requested_outputs.append(metrics)

    # Initialise a counter for the OutputDefinitions
    counter = 1

    # Set location to write regional exp files
    asset_dir = '/g/data/au88/lb9857/gitRepos/ACCESS-AIS3/assets'
    
    # Iterate over all rows of the imbie geodataframe
    for idx, row in imbie.iterrows():

        ## Generate an exp file to be used with ContourToMesh
        ## ---------------------------------------------------
        
        # Isolate one basin (keeping as GeoDataFrame)
        basin_gdf = imbie.iloc[[idx]]
    
        # Extract the name
        basin_name = basin_gdf.iloc[0]["NAME"]

        print(f"-- Generating *.exp file for basin {basin_name}...")
        # Form exp filepath
        exp_file = f"{asset_dir}/region_{basin_name}.exp"
    
        # Save to exp file
        pyissm.tools.exp.gdf_to_exp(basin_gdf, exp_file)
    
        ## Generate mask on model mesh
        ## NOTE: This is not necessary here. You can use `maskexpstring` in `regionaloutput` below and this
        ## step is conducted when the model file is marshalled. However the mask is not saved in the netCDF.
        ## For completeness (in case exp files are removed), we interpolate it here and feed the mask directly
        ## below.
        ## ---------------------------------------------------
        print(f"-- Interpolating basin {basin_name} onto model mesh...")
        
        basin_mask = pyissm.tools.wrappers.ContourToMesh(index = md.mesh.elements,
                                                         x = md.mesh.x,
                                                         y = md.mesh.y,
                                                         contour_name = exp_file,
                                                         interp_type = 'node',
                                                         edge_value = 1)

        ## Generate unique OutputDefinitions for all regions and metrics
        ## ---------------------------------------------------
        print(f"-- Compiling regionaloutput definition for {basin_name}...")

        for metric in metrics:

            # Define unique name
            output_name = f"{basin_name}_{metric}"

            # Compile the regionaloutput
            output = pyissm.model.classes.regionaloutput(
                name = output_name,
                outputnamestring = metric,
                mask = basin_mask,
                definitionstring = f"Outputdefinition{counter}")

            # Append the output the definitions list
            definitions.append(output)

            # Append the output_name to the requested_outputs
            requested_outputs.append(output_name)

            # Increment the counter
            counter += 1

    # Assign the lists to the model
    md.outputdefinition.definitions = definitions
    md.transient.requested_outputs = requested_outputs

# Save model
## NOTE: This fails for now. `save_model()` does not support lists of classes yet. This needs to be updated to
## support this workflow.
print(f"-- Saving model to file...")
pyissm.model.io.save_model(f'{model_dir}/AIS3_regional_outputs.nc')

In [ ]:
if 'ho_relax' in steps:

    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_regional_outputs.nc')

    # Define timestepping
    md.timestepping.start_time = 0
    md.timestepping.time_step = 1/12
    md.timestepping.final_time = 100

    # Set transient settings
    md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
    md.transient.issmb = 1
    md.transient.ismasstransport = 1
    md.transient.isstressbalance = 1
    md.transient.isgroundingline = 1

    # Disable inversion
    md.inversion.iscontrol = 0

    # Update cluster
    md.cluster = cluster
    os.makedirs(f'{execution_dir}/ho_relax', exist_ok = True)
    md.cluster.executionpath = f'{execution_dir}/ho_relax'

    # Execute / Save
    if load_only:
        
        md = pyissm.model.execute.solve(md, 'transient', runtime_name = False, load_only = True)

        print(f"-- Saving model to file...")
        ## NOTE: This fails for now. `save_model()` does not support lists of classes yet. This needs to be updated to
        ## support this workflow.
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_ho_relax.nc')

    else:
        md = pyissm.model.execute.solve(md, 'transient', runtime_name = False, load_only = False)
